# 01 — Batch Risk Scoring Pipeline

Enterprise pattern: train a readmission risk model and run batch inference
over a patient population, producing structured risk scores for downstream
consumption (dashboards, alerts, clinical decision support).

Uses the `pyhealth_enterprise` package throughout — this is what production code looks like.

In [ ]:
from pathlib import Path
from pyhealth_enterprise.config import settings
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth_enterprise.tasks.readmission import setup_readmission_task
from pyhealth_enterprise.models.registry import ModelName, get_model
from pyhealth_enterprise.pipelines.batch_risk_scorer import BatchRiskScorer, RiskScoreConfig
from pyhealth_enterprise.pipelines.report_generator import ReportGenerator

# Load dataset
ds = SyntheticEHRDataset()
ds.load()
print(f"Loaded {ds.get_patient_count()} patients, {ds.get_visit_count()} visits")

In [ ]:
# Set up readmission task and loaders
train_loader, val_loader, test_loader = setup_readmission_task(
    ds.dataset, batch_size=32
)

# Get task dataset for model instantiation
from pyhealth.tasks import readmission_prediction_mimic3_fn
task_dataset = ds.dataset.set_task(readmission_prediction_mimic3_fn)

In [ ]:
# Configure and train the risk scorer
model = get_model(
    ModelName.RETAIN,
    task_dataset,
    feature_keys=["conditions", "drugs"],
    label_key="readmission",
)

config = RiskScoreConfig(
    epochs=50,
    monitor="pr_auc",
    threshold_high=0.7,
    threshold_medium=0.4,
)
scorer = BatchRiskScorer(model, model_name="retain_readmission_v1", config=config)
train_metrics = scorer.train(train_loader, val_loader)
print("Training complete:", train_metrics)

In [ ]:
# Score the test population
risk_df = scorer.score_batch(test_loader)
print(f"Scored {len(risk_df)} patients")
risk_df.head(10)

In [ ]:
# Risk distribution
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(risk_df["risk_score"], bins=20, ax=axes[0])
axes[0].set_title("Risk Score Distribution")
axes[0].set_xlabel("Risk Score")

risk_df["risk_label"].value_counts().plot.bar(ax=axes[1], color=["#c00", "#e65c00", "#2a7a2a"])
axes[1].set_title("Risk Level Distribution")
axes[1].set_xlabel("Risk Level")

plt.tight_layout()
plt.show()

In [ ]:
# Export CSV and HTML report
output_dir = settings.PROJECT_ROOT / "data" / "processed"

scorer.export_csv(risk_df, output_dir / "risk_scores.csv")

report_gen = ReportGenerator()
report_gen.generate_risk_summary(risk_df, output_dir / "risk_summary.html")
report_gen.generate_risk_excel(risk_df, output_dir / "risk_scores.xlsx")

print(f"Outputs written to: {output_dir}")